# Dimension 1: Development effort and Maintainability

## Environment setup

Installs the metric libraries used later in the notebook and restarts the Python runtime so the packages are available.


In [0]:
%pip install radon==6.0.1 lizard==1.17.10
dbutils.library.restartPython()

## Imports and pipeline configuration

Defines dependencies, workspace paths for both pipeline paradigms, and shared chart colors.


In [0]:
# Import dependencies and define the global notebook configuration.

import os
import pandas as pd
from radon.complexity import cc_visit
from radon.metrics import mi_visit
from radon.raw import analyze as raw_analyze
import lizard
import matplotlib.pyplot as plt
import numpy as np


# Root folders for both pipeline implementations in the Databricks workspace.
DECLARATIVE_ROOT = "/Workspace/Users/klementf.wwi23@student.dhbw-heidenheim.de/thesis-databricks-pipeline-poc/declarative_pipeline"
IMPERATIVE_ROOT  = "/Workspace/Users/klementf.wwi23@student.dhbw-heidenheim.de/thesis-databricks-pipeline-poc/imperative_pipeline"

PIPELINES = {
    "declarative": DECLARATIVE_ROOT,
    "imperative": IMPERATIVE_ROOT,
}

declarative_color = "#0F2DB3"
imperative_color = "#0072EF"

## File metadata helpers

Provides small helper functions to derive the medallion layer, job name, and data quality flag from each file path.


In [0]:
# Helper functions for deriving pipeline metadata from file paths.

def detect_layer_from_filename(path: str) -> str:
    base = os.path.basename(path)
    if base.startswith("01_"):
        return "bronze"
    elif base.startswith("02_"):
        return "silver"
    elif base.startswith("03_"):
        return "gold"
    else:
        return "unknown"


def is_data_quality(path: str) -> bool:
    return "data_quality" in path.replace("\\", "/")


def detect_job_name(path: str) -> str:
    base = os.path.basename(path)
    name, _ = os.path.splitext(base)
    return name

## Radon file analysis helper

Wraps Radon calls to calculate SLOC, cyclomatic complexity, and maintainability metrics for a single Python file.


In [0]:
# Helper function for analyzing one Python file with Radon.

def analyze_python_file(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        code = f.read()

    # Raw metrics
    raw = raw_analyze(code)
    sloc = raw.sloc
    lloc = raw.lloc

    # Cyclomatic Complexity
    cc_results = cc_visit(code)
    if cc_results:
        cc_scores = [item.complexity for item in cc_results]
        avg_cc = sum(cc_scores) / len(cc_scores)
        max_cc = max(cc_scores)
        cc_count = len(cc_scores)
    else:
        avg_cc = 0.0
        max_cc = 0.0
        cc_count = 0

    # Maintainability Index
    mi_results = mi_visit(code, multi=True)

    # mi_results 
    mi_scores = []
    if isinstance(mi_results, (int, float)):
        mi_scores = [float(mi_results)]
    elif mi_results:
        first = mi_results[0]
        if hasattr(first, "mi"):
            mi_scores = [r.mi for r in mi_results]
        else:
            mi_scores = [float(r) for r in mi_results]

    if mi_scores:
        avg_mi = sum(mi_scores) / len(mi_scores)
        min_mi = min(mi_scores)
        mi_count = len(mi_scores)
    else:
        avg_mi = 100.0
        min_mi = 100.0
        mi_count = 0

    return {
        "sloc": sloc,
        "lloc": lloc,
        "avg_cc": avg_cc,
        "max_cc": max_cc,
        "cc_entities": cc_count,
        "avg_mi": avg_mi,
        "min_mi": min_mi,
        "mi_entities": mi_count,
    }
     

## Discover pipeline source files

Scans both implementations, records metadata for every Python file, and separates business transformations from data quality jobs.


In [0]:
# Collect all Python files from both pipelines and enrich them with metadata.

records = []

for paradigm_type, root in PIPELINES.items():
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.endswith(".py"):
                continue

            full_path = os.path.join(dirpath, fname)

            records.append({
                "paradigm": paradigm_type,
                "file_path": full_path,
                "layer": detect_layer_from_filename(full_path),
                "job": detect_job_name(full_path),
                "is_data_quality": is_data_quality(full_path),
            })

df_files = pd.DataFrame(records)

print("All discovered .py files:")
display(df_files)

print("Business transformation jobs only, excluding data_quality:")
df_main_jobs = df_files[~df_files["is_data_quality"]].reset_index(drop=True)
display(df_main_jobs)

## Calculate file-level code metrics

Runs the Radon helper for each discovered file and builds the base metrics table.


In [0]:
# Calculate Radon metrics for every discovered Python file.

metrics_records = []

for _, row in df_files.iterrows():
    path = row["file_path"]
    try:
        metrics = analyze_python_file(path)
    except Exception as e:
        print(f"Error while analyzing {path}: {e}")
        continue

    record = {
        "paradigm": row["paradigm"],
        "layer": row["layer"],
        "job": row["job"],
        "file_path": row["file_path"],
        "is_data_quality": row["is_data_quality"],
    }
    record.update(metrics)
    metrics_records.append(record)

df_metrics_file = pd.DataFrame(metrics_records)

print("File-level Radon metrics:")
display(df_metrics_file)

## Detect duplicate code blocks

Uses Lizard to identify repeated code blocks and aggregates duplicate counts per job.


In [0]:
# Analyze duplicated code blocks with Lizard and handle empty results.

dup_records = []

for paradigm_type, root in PIPELINES.items():
    print(f"Lizard-Analyse for Pipeline {paradigm_type}: {root}")
    result = lizard.analyze(root)

    duplicate_groups = getattr(result, "crossfile_duplication", None) \
        or getattr(result, "cross_file_duplicate", None) \
        or []

    for dup_group in duplicate_groups:
        files = getattr(dup_group, "files", [])
        for fi in files:
            file_path = fi.filename
            start_line = fi.start_line
            end_line = fi.end_line

            match = df_files[df_files["file_path"] == file_path]
            if match.empty:
                continue

            meta = match.iloc[0]

            dup_records.append({
                "paradigm": paradigm_type,
                "file_path": file_path,
                "layer": meta["layer"],
                "job": meta["job"],
                "is_data_quality": meta["is_data_quality"],
                "dup_block_id": id(dup_group),
                "start_line": start_line,
                "end_line": end_line,
            })

# Build the duplicate-block DataFrame from detected records.
if dup_records:
    df_dups = pd.DataFrame(dup_records)
    print("Detected duplicate blocks, raw Pandas DataFrame:")
    display(df_dups)
else:
    # Print an informational message instead of displaying an empty DataFrame.
    df_dups = pd.DataFrame(
        columns=[
            "paradigm",
            "file_path",
            "layer",
            "job",
            "is_data_quality",
            "dup_block_id",
            "start_line",
            "end_line",
        ]
    )
    print("No duplicate blocks found; df_dups is empty.")

# Aggregate duplicate blocks when Lizard returned entries.
if not df_dups.empty:
    df_dup_counts = (
        df_dups.groupby(["paradigm", "layer", "job", "is_data_quality"], as_index=False)["dup_block_id"]
        .nunique()
        .rename(columns={"dup_block_id": "duplicated_blocks"})
    )
    print("Number of duplicated blocks per job, Pandas DataFrame:")
    display(df_dup_counts)
else:
    df_dup_counts = pd.DataFrame(
        columns=["paradigm", "layer", "job", "is_data_quality", "duplicated_blocks"]
    )
    print("No duplicates found; df_dup_counts is empty.")

## Combine code metrics and duplication metrics

Merges Radon metrics with duplicate-code counts and creates the main analysis table.


In [0]:
# Combine Radon code metrics and duplicate-code counts into one DataFrame.

# Ensure that the file-level Radon metrics DataFrame exists.
if "df_metrics_file" not in globals():
    # Raise a clear error if the required metrics cell has not been executed.
    raise RuntimeError(
        "df_metrics_file is not defined. "
        "Please run the cell that calculates file-level Radon metrics first."
    )

# Create an empty duplicate-count DataFrame if the Lizard cell was skipped or failed.
if "df_dup_counts" not in globals():
    df_dup_counts = pd.DataFrame(
        columns=["paradigm", "layer", "job", "is_data_quality", "duplicated_blocks"]
    )

# Merge Radon metrics with duplicate-code counts.
df_all = df_metrics_file.merge(
    df_dup_counts,
    on=["paradigm", "layer", "job", "is_data_quality"],
    how="left"
)

# Treat missing duplicate-code values as zero.
df_all["duplicated_blocks"] = df_all["duplicated_blocks"].fillna(0).astype(int)

print("All metrics, including duplicates, at job level:")
display(df_all)

print("Business transformation jobs only, excluding data_quality, with all metrics:")
df_all_main = df_all[~df_all["is_data_quality"]]
display(df_all_main)

## Aggregate metrics by paradigm and layer

Summarizes business transformation metrics by paradigm and medallion layer for later comparison.


In [0]:
# Aggregate metrics by paradigm and medallion layer for evaluation.

df_summary = (
    df_all_main
    .groupby(["paradigm", "layer"], as_index=False)
    .agg({
        "sloc": "mean",
        "lloc": "mean",
        "avg_cc": "mean",
        "avg_mi": "mean",
        "duplicated_blocks": "mean",
    })
    .rename(columns={
        "sloc": "avg_sloc_per_job",
        "lloc": "avg_lloc_per_job",
        "avg_cc": "avg_cc_per_job",
        "avg_mi": "avg_mi_per_job",
        "duplicated_blocks": "avg_dup_blocks_per_job",
    })
    .sort_values(["paradigm", "layer"])
)

print("Summary by paradigm and layer, transformation jobs only:")
display(df_summary)

## Create pipeline-level summaries

Builds separate summaries for all jobs, transformation jobs only, and data quality jobs only.


In [0]:
# These summaries compare the two paradigms at different scopes: all jobs, transformation jobs, and DQ jobs.

import pandas as pd

if "df_all" not in globals():
    raise RuntimeError(
        "df_all is not defined. "
        "Please run the combination cell for Radon metrics and duplicate counts first."
    )

# 1) All Jobs
df_summary_all = (
    df_all
    .groupby(["paradigm"], as_index=False)
    .agg({
        "sloc": "mean",
        "lloc": "mean",
        "avg_cc": "mean",
        "avg_mi": "mean",
        "duplicated_blocks": "mean",
    })
    .rename(columns={
        "sloc": "avg_sloc_per_job",
        "lloc": "avg_lloc_per_job",
        "avg_cc": "avg_cc_per_job",
        "avg_mi": "avg_mi_per_job",
        "duplicated_blocks": "avg_dup_blocks_per_job",
    })
)

df_summary_all = df_summary_all.round(2)

print("Pipeline summary for all jobs, including data_quality:")
display(df_summary_all)


# 2) Only transformation jobs
df_all_main = df_all[~df_all["is_data_quality"]]

df_summary_main = (
    df_all_main
    .groupby(["paradigm"], as_index=False)
    .agg({
        "sloc": "mean",
        "lloc": "mean",
        "avg_cc": "mean",
        "avg_mi": "mean",
        "duplicated_blocks": "mean",
    })
    .rename(columns={
        "sloc": "avg_sloc_per_job",
        "lloc": "avg_lloc_per_job",
        "avg_cc": "avg_cc_per_job",
        "avg_mi": "avg_mi_per_job",
        "duplicated_blocks": "avg_dup_blocks_per_job",
    })
)

df_summary_main = df_summary_main.round(2)

print("Pipeline summary for transformation jobs only, excluding data_quality:")
display(df_summary_main)


# 3) Only data_quality jobs
df_all_dq = df_all[df_all["is_data_quality"]]

df_summary_dq = (
        df_all_dq
        .groupby(["paradigm"], as_index=False)
        .agg({
            "sloc": "mean",
            "lloc": "mean",
            "avg_cc": "mean",
            "avg_mi": "mean",
            "duplicated_blocks": "mean",
        })
        .rename(columns={
            "sloc": "avg_sloc_per_job",
            "lloc": "avg_lloc_per_job",
            "avg_cc": "avg_cc_per_job",
            "avg_mi": "avg_mi_per_job",
            "duplicated_blocks": "avg_dup_blocks_per_job",
        })
    )

df_summary_dq = df_summary_dq.round(2)

print("Pipeline summary for data_quality jobs only:")
display(df_summary_dq)

## Visualize development code metrics

Generates comparison charts for SLOC, cyclomatic complexity, and maintainability index.


In [0]:
# Create comparison figures for the development code metrics.

required_dfs = ["df_summary_main", "df_summary_dq", "df_summary_all"]
for name in required_dfs:
    if name not in globals():
        raise RuntimeError(
            f"{name} is not defined. "
            "Please run the cells that create the summary DataFrames first."
        )

def extract_segment_values(kpi_col: str):
    main_dec = df_summary_main.loc[df_summary_main["paradigm"] == "declarative", kpi_col].iloc[0]
    main_imp = df_summary_main.loc[df_summary_main["paradigm"] == "imperative", kpi_col].iloc[0]

    dq_dec = df_summary_dq.loc[df_summary_dq["paradigm"] == "declarative", kpi_col].iloc[0]
    dq_imp = df_summary_dq.loc[df_summary_dq["paradigm"] == "imperative", kpi_col].iloc[0]

    all_dec = df_summary_all.loc[df_summary_all["paradigm"] == "declarative", kpi_col].iloc[0]
    all_imp = df_summary_all.loc[df_summary_all["paradigm"] == "imperative", kpi_col].iloc[0]

    return (
        [main_dec, dq_dec, all_dec],
        [main_imp, dq_imp, all_imp]
    )

segments = ["Transformation", "Data quality", "All jobs"]
x = np.arange(len(segments))
width = 0.35


# --- Figure 1: Average SLOC per job ---
lloc_dec, lloc_imp = extract_segment_values("avg_lloc_per_job")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width/2, lloc_dec, width, label="Declarative", color=declarative_color)
ax.bar(x + width/2, lloc_imp, width, label="Imperative", color=imperative_color)
ax.set_xticks(x)
ax.set_xticklabels(segments)
ax.set_ylabel("Average LLOC per job")
#ax.set_ylim(20, None)
ax.legend()

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()


# --- Figure 2: Average maintainability index ---
mi_dec, mi_imp = extract_segment_values("avg_mi_per_job")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width/2, mi_dec, width, label="Declarative", color=declarative_color)
ax.bar(x + width/2, mi_imp, width, label="Imperative", color=imperative_color)
ax.set_xticks(x)
ax.set_xticklabels(segments)
ax.set_ylabel("Average maintainability index")
ax.set_ylim(50, None)
ax.legend()

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()


# --- Figure 3: Average cyclomatic complexity ---
cc_dec, cc_imp = extract_segment_values("avg_cc_per_job")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width/2, cc_dec, width, label="Declarative", color=declarative_color)
ax.bar(x + width/2, cc_imp, width, label="Imperative", color=imperative_color)
ax.set_xticks(x)
ax.set_xticklabels(segments)
ax.set_ylabel("Average cyclomatic complexity")
ax.set_ylim(1.0, 1.5)
ax.legend()

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()
